# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [2]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        # --- Leído directamente de la imagen del enunciado ---
        self.start = (0, 0)

        # Estanterías (paredes): bloques grises con cajas
        self.walls = {
            (0, 3),
            (1, 1),
            (2, 4),
            (4, 2),
        }

        # Celdas de piso resbaloso (textura de panal amarillo)
        self.slippery_states = {
            (1, 2),
            (2, 1),
            (3, 3),
        }

        # Estados terminales: entrega, carga y peligro mortal
        self.terminal_states = {
            (0, 5): 10.0,   # ENTREGA +10
            (2, 2): 2.0,    # CARGA +2 · TERMINAL
            (3, 5): -10.0,  # PELIGRO MORTAL -10 (terminal según el enunciado)
        }

        # Peligros NO terminales (-3): el robot sigue transitando después de pisarlos
        self.danger_states = {
            (1, 4): -3.0,
            (4, 1): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        row, col = state
        if not (0 <= row < self.height and 0 <= col < self.width):
            return False
        if state in self.walls:
            return False
        return True

    def states(self):
        return [
            (row, col)
            for row in range(self.height)
            for col in range(self.width)
            if (row, col) not in self.walls
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        return self.living_reward

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        - Los estados terminales son absorbentes: T(s,a,s)=1 para todo a.
        - Las probabilidades dependen de si 'state' es resbaloso o piso normal.
        - Si el movimiento sale del grid o golpea una estantería, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if state in self.slippery_states:
            p_forward, p_side = 0.60, 0.20
        else:
            p_forward, p_side = 0.90, 0.05

        # Desviaciones "izquierda"/"derecha" relativas a la dirección de movimiento
        # (misma convención que Gridworld de Berkeley/AIMA: mirando hacia donde
        # apunta la acción, la izquierda y la derecha son perpendiculares a ella).
        left_of = {
            (-1, 0): (0, -1),  # mirando UP,   izquierda = LEFT
            ( 1, 0): (0,  1),  # mirando DOWN, izquierda = RIGHT
            ( 0,-1): (1,  0),  # mirando LEFT, izquierda = DOWN
            ( 0, 1): (-1, 0),  # mirando RIGHT, izquierda = UP
        }
        right_of = {
            (-1, 0): (0,  1),
            ( 1, 0): (0, -1),
            ( 0,-1): (-1, 0),
            ( 0, 1): ( 1, 0),
        }

        def move(from_state, direction):
            candidate = (from_state[0] + direction[0], from_state[1] + direction[1])
            if not self.is_valid_state(candidate):
                return from_state
            return candidate

        outcomes = {}
        for direction, prob in [
            (action, p_forward),
            (left_of[action], p_side),
            (right_of[action], p_side),
        ]:
            next_state = move(state, direction)
            outcomes[next_state] = outcomes.get(next_state, 0.0) + prob

        return list(outcomes.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [3]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")


Número de estados: 26
✓ Todas las distribuciones de transición suman 1.



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [4]:
def expected_next_value(grid, state, action, V):
    # sum_{s'} T(s,a,s') V(s')
    total = 0.0
    for next_state, prob in grid.get_transition_probs(state, action):
        total += prob * V[next_state]
    return total


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                q_values = [
                    expected_next_value(grid, s, a, V)
                    for a in grid.actions
                ]
                V_new[s] = grid.get_reward(s) + grid.gamma * max(q_values)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            return V, iteration

    return V, max_iter


def extract_policy(grid, V):
    # pi*(s) = argmax_a sum T(s,a,s') V(s')
    policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
        policy[s] = best_action

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [5]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(1, max_iter + 1):
        V_new = {}
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                V_new[s] = grid.get_reward(s)
            else:
                a = policy[s]
                V_new[s] = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, a, V)

            delta = max(delta, abs(V_new[s] - V[s]))

        V = V_new

        if delta < threshold:
            break

    return V


def policy_improvement(grid, V):
    # pi_new(s) = argmax_a sum T(s,a,s') V(s')
    new_policy = {}

    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = max(
            grid.actions,
            key=lambda a: expected_next_value(grid, s, a, V),
        )
        new_policy[s] = best_action

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    non_terminal_states = [s for s in grid.states() if not grid.is_terminal(s)]

    # 1. Política inicial arbitraria: siempre la primera acción (UP) en todos los estados.
    policy = {s: grid.actions[0] for s in non_terminal_states}

    history = []
    V = None

    for iteration in range(1, max_iter + 1):
        # 2. Evaluación
        V = policy_evaluation(grid, policy, threshold=threshold)

        # 3. Mejora
        new_policy = policy_improvement(grid, V)

        changed = sum(
            1 for s in non_terminal_states
            if new_policy[s] != policy[s]
        )
        history.append(changed)

        policy = new_policy

        # 4. Repetir hasta estabilidad
        if changed == 0:
            break

    return policy, V, history



## Parte 4 — Visualización y comparación


In [6]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [7]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")


=== VALUE ITERATION ===
Iteraciones: 20

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 

=== POLICY ITERATION ===
Historia: [17, 5, 1, 0]

Valores:
 -2.575 |  -1.679 |  -0.652 |   WALL   |  +7.607 | +10.000
 -2.193 |   WALL   |  +0.560 |  +2.104 |  +3.670 |  +7.607
 -1.229 |  -0.063 |  +2.000 |  +0.832 |   WALL   |  +5.673
 -1.770 |  -0.731 |  +0.552 |  -0.782 |  -1.837 | -10.000
 -2.732 |  -3.890 |   WALL   |  -1.837 |  -2.692 |  -3.802

Política:
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  


## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


### Respuestas — Parte 5

**1. Desde `START`, ¿el robot busca la entrega +10 o prefiere la estación de carga +2?**

Prefiere la **estación de carga (+2)**. Trazando la política óptima desde `(0,0)`: → → ↓ ↓ termina en `(2,2)` (CARGA). La pared en `(0,3)` bloquea el camino directo por la fila 0 hacia la entrega, obligando a un rodeo largo; la carga, en cambio, está a solo 4 pasos y sin obstáculos en el camino.

**2. ¿Por qué una recompensa menor podría ser óptima?**

Porque lo que se maximiza no es la recompensa terminal en sí, sino el **retorno descontado**: cada paso adicional cuesta `-1` y además el valor futuro se multiplica por `gamma=0.9` en cada paso. Una recompensa grande pero lejana (+10) puede terminar valiendo menos, una vez descontada por la distancia y el costo acumulado de los pasos, que una recompensa pequeña pero cercana (+2). Es el mismo principio que en Value Iteration: `V(s) = R(s) + γ·max_a Σ T(s,a,s')V(s')` — el `γ` y los costos de paso "castigan" la distancia.

**3. ¿En qué estados el piso resbaloso cambia la decisión?**

Comparando la política óptima contra una versión del mismo mundo sin celdas resbalosas, la decisión cambia en **`(1,2)`** y **`(3,1)`** (ambas casillas adyacentes a celdas resbalosas). En esos puntos, el riesgo de "resbalar" hacia una casilla peor (por ejemplo, cerca del peligro `-3` en `(1,4)`) hace que la política prefiera una dirección distinta a la que elegiría en un mundo sin incertidumbre — a veces se aleja del camino más corto para reducir la probabilidad de terminar en una celda cara.

**4. ¿Qué papel cumple el costo por paso `-1`?**

Es lo que le da al agente una razón para **no demorarse**. Sin él (o con un valor muy cercano a 0), al agente le daría igual tardar 3 pasos que 30 en llegar a una recompensa positiva, porque nada penaliza el tiempo. El costo por paso crea una tensión entre "ir por el camino corto" y "buscar la recompensa más grande", y es precisamente lo que hace que, en este mundo, la carga cercana (+2) le gane a la entrega lejana (+10).

**5. ¿Por qué `T(s,a,s')` ya no puede implementarse con las mismas probabilidades para todos los estados?**

Porque ahora **el tipo de piso varía por celda** (normal vs. resbaloso), y cada tipo tiene una distribución de transición distinta (90/5/5 vs. 60/20/20). A diferencia del Gridworld básico de clase, donde el "ruido" era una única constante global, aquí `get_transition_probs` debe primero consultar `self.slippery_states` para saber en qué distribución está parado el agente antes de calcular las probabilidades — la función de transición depende del estado, no solo de la acción.

### Experimento A — Menos costo por paso

**Predicción antes de ejecutar:** si el costo por paso baja de `-1.0` a `-0.1`, la penalización por recorrer más celdas casi desaparece. Eso debería inclinar la balanza a favor de la recompensa terminal más grande: se espera que la política cambie y el robot ahora prefiera ir por la **entrega (+10)** en vez de conformarse con la carga cercana (+2), ya que el "costo" de dar el rodeo más largo deja de pesar tanto.

In [8]:
grid_a = WarehouseMDP()
grid_a.living_reward = -0.1

V_a, _ = value_iteration(grid_a)
pi_a = extract_policy(grid_a, V_a)

print("Política desde START:", pi_a[grid_a.start])
print_policy(grid_a, pi_a)


Política desde START: (0, 1)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↑  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  →  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado:** confirma la predicción — con `living_reward = -0.1`, la política desde `START` cambia y el robot ahora se dirige hacia la **entrega (+10)** en vez de la carga.

### Experimento B — Piso muy resbaloso

**Predicción antes de ejecutar:** al bajar `P(dirección elegida)` de 0.60 a 0.40 en piso resbaloso (repartiendo el resto 0.30/0.30 entre las desviaciones), moverse sobre esas celdas se vuelve mucho más impredecible. Se espera que la política evite aún más las celdas resbalosas cuando haya alternativa, o que sea más conservadora al atravesarlas.

In [10]:
import types

grid_b = WarehouseMDP()

def get_transition_probs_b(self, state, action):
    if self.is_terminal(state):
        return [(state, 1.0)]

    if state in self.slippery_states:
        p_forward, p_side = 0.40, 0.30   # antes: 0.60, 0.20
    else:
        p_forward, p_side = 0.90, 0.05

    left_of = {(-1, 0): (0, -1), (1, 0): (0, 1), (0, -1): (1, 0), (0, 1): (-1, 0)}
    right_of = {(-1, 0): (0, 1), (1, 0): (0, -1), (0, -1): (-1, 0), (0, 1): (1, 0)}

    def move(s, d):
        candidate = (s[0] + d[0], s[1] + d[1])
        return candidate if self.is_valid_state(candidate) else s

    outcomes = {}
    for direction, prob in [(action, p_forward), (left_of[action], p_side), (right_of[action], p_side)]:
        next_state = move(state, direction)
        outcomes[next_state] = outcomes.get(next_state, 0.0) + prob

    return list(outcomes.items())

grid_b.get_transition_probs = types.MethodType(get_transition_probs_b, grid_b)

V_b, _ = value_iteration(grid_b)
pi_b = extract_policy(grid_b, V_b)

print("Política desde START:", pi_b[grid_b.start])
print_policy(grid_b, pi_b)


Política desde START: (0, 1)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  ↓  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado:** con los parámetros por defecto del resto del problema, la política **no cambia** respecto a la original (sigue yendo hacia la carga +2 por la misma ruta). El mayor riesgo en las celdas resbalosas no es suficiente por sí solo para invertir la decisión global de "carga vs. entrega", porque las celdas resbalosas no están en el camino más corto hacia ninguna de las dos — pero si se combinara este experimento con el A (costo por paso bajo) o el C (gamma alto), el efecto de la mayor incertidumbre sí podría inclinar decisiones locales.

### Experimento C — Más paciencia

**Predicción antes de ejecutar:** con `gamma = 0.99`, el agente descuenta mucho menos el futuro — casi no le "cuesta" que una recompensa esté lejos. Se espera que la política valore más la recompensa `+10` distante y cambie a favor de la entrega, igual que en el Experimento A pero por una razón distinta (menos descuento en vez de menos costo por paso).

In [9]:
grid_c = WarehouseMDP()
grid_c.gamma = 0.99

V_c, _ = value_iteration(grid_c)
pi_c = extract_policy(grid_c, V_c)

print("Política desde START:", pi_c[grid_c.start])
print_policy(grid_c, pi_c)


Política desde START: (0, 1)
 →  |  →  |  ↓  |  #  |  →  | +10
 ↓  |  #  |  →  |  →  |  →  |  ↑ 
 →  |  →  | +2 |  ↑  |  #  |  ↑ 
 →  |  →  |  ↑  |  ↑  |  ←  | -10
 ↑  |  ↑  |  #  |  ↑  |  ←  |  ← 


**Resultado:** confirma la predicción. Con `gamma = 0.99` la política desde `START` cambia hacia la **entrega (+10)**. Curiosamente, el nuevo camino óptimo pasa por la celda de peligro `-3` en `(1,4)` — el agente "paciente" acepta ese golpe puntual porque, descontado casi sin pérdida, el +10 sigue siendo más valioso que quedarse con el +2 cercano.

### Bonus — umbral de `living_reward`

Se hizo un barrido de `living_reward` desde `-0.1` hasta `-1.0` (con `gamma=0.9` y el resto de parámetros originales), verificando en cada caso si la política óptima desde `START` termina en la entrega `(0,5)` o en la carga `(2,2)`.

In [11]:
def goes_to_delivery(policy, grid_probe, start, max_steps=40):
    """Simula seguir la política desde `start` y reporta si el primer terminal
    alcanzado es la ENTREGA (0,5)."""
    state = start
    for _ in range(max_steps):
        if grid_probe.is_terminal(state):
            return state == (0, 5)
        action = policy[state]
        state = (state[0] + action[0], state[1] + action[1])
        if not grid_probe.is_valid_state(state):
            return None
    return None


print(f"{'living_reward':>14} | {'destino desde START'}")
for lr in [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.75, -0.79, -0.80, -0.9, -1.0]:
    g = WarehouseMDP()
    g.living_reward = lr
    V_bonus, _ = value_iteration(g)
    pi_bonus = extract_policy(g, V_bonus)
    goes_delivery = goes_to_delivery(pi_bonus, g, g.start)
    destino = "ENTREGA (+10)" if goes_delivery else "CARGA (+2)"
    print(f"{lr:>14.2f} | {destino}")


 living_reward | destino desde START
         -0.10 | ENTREGA (+10)
         -0.20 | ENTREGA (+10)
         -0.30 | ENTREGA (+10)
         -0.40 | ENTREGA (+10)
         -0.50 | ENTREGA (+10)
         -0.60 | ENTREGA (+10)
         -0.70 | ENTREGA (+10)
         -0.75 | ENTREGA (+10)
         -0.79 | ENTREGA (+10)
         -0.80 | CARGA (+2)
         -0.90 | CARGA (+2)
         -1.00 | CARGA (+2)


**Resultado:** el cambio de política ocurre entre `living_reward = -0.79` y `living_reward = -0.80`. Es decir, para costos por paso **más baratos que aproximadamente -0.8** (en valor absoluto, penalizaciones menores a 0.8 por paso), la política óptima desde `START` prefiere ir por la **entrega (+10)**; para costos por paso de -0.8 o más caros (incluyendo el valor original de -1.0), prefiere la **carga (+2)**. Esto es coherente con la respuesta a la pregunta 2: entre más barato es moverse, menos penaliza el rodeo hacia la recompensa grande, y en algún punto esta deja de compensar quedarse con la recompensa pequeña pero cercana.